# Customers, invoices and relationship paths

Import a product catalogue, customers and dated invoices from CSV. Import quantities, prices and totals, then read customer and product fields through relationship paths.

[Watch lesson 2](https://asqueel.org/learn/02-installation-first-db/)

**Run the cells from top to bottom in a fresh Python kernel.** Each notebook includes its own starting state. SQLite runs in memory; restarting the kernel discards the database. To repeat the exercise, restart the kernel and run all cells. Colab execution requires a Google account.

© 2026 Softwell S.r.l. · Apache-2.0

## Before the examples: `_ = ...` is optional

It assigns a return value we do not need, preventing its automatic display in the Python console or notebook. The call still runs normally. This is a Python convention, not an Asqueel requirement. When you need the result, use a meaningful variable name.

## Prepare the notebook

The video starts in a terminal. Here, install the same official package in the notebook kernel. No virtual environment or separate Python console is needed in Colab.

In [ ]:
%pip install -q "asqueel[migration]==0.6.0"

In [ ]:
%config InteractiveShell.ast_node_interactivity = "all"

## Recreate the starting point

The previous lessons are rebuilt below. The four fictional CSV files are included in this notebook and written into a new temporary folder. No downloads or existing database are needed.

In [ ]:
#@title Prepare the sample CSV files
from pathlib import Path
import os
import tempfile
_lesson_folder = tempfile.TemporaryDirectory(prefix="asqueel-notebook-")
os.chdir(_lesson_folder.name)
_csv_files = {'customers.csv': 'id,name,city\n1,Ada,Florence\n2,Grace,London\n3,Linus,Helsinki\n4,Guido,Amsterdam\n5,Elena,Florence\n6,Alan,Manchester\n7,Margaret,New York\n8,Barbara,Boston\n', 'invoice_rows.csv': 'id,invoice_id,product_code,quantity,unit_price,total\n1,101,TEA,2,3.50,7.00\n2,101,COF,1,6.90,6.90\n3,101,MUG,4,5.25,21.00\n4,102,BTEA,2,3.20,6.40\n5,102,HTEA,3,4.10,12.30\n6,102,HON,4,7.50,30.00\n7,103,SUG,3,2.30,6.90\n8,103,CHO,4,3.90,15.60\n9,103,BIS,1,2.80,2.80\n10,104,JAM,4,4.60,18.40\n11,104,OAT,1,2.40,2.40\n12,104,JUI,2,2.70,5.40\n13,105,WAT,1,0.90,0.90\n14,105,LEM,2,1.80,3.60\n15,105,KET,3,32.90,98.70\n16,106,POT,2,18.50,37.00\n17,106,PLT,3,6.40,19.20\n18,106,BWL,4,7.80,31.20\n19,107,SPO,3,9.50,28.50\n20,107,TRY,4,14.90,59.60\n21,107,JAR,1,8.20,8.20\n22,108,NAP,4,12.00,48.00\n23,108,TWL,1,5.60,5.60\n24,108,FLT,2,2.10,4.20\n25,109,TIN,1,6.70,6.70\n26,109,MAT,2,11.40,22.80\n27,109,CAF,3,29.90,89.70\n28,110,BOT,2,13.50,27.00\n29,110,STR,3,4.80,14.40\n30,110,SCL,4,24.90,99.60\n31,111,TEA,3,3.50,10.50\n32,111,COF,4,6.90,27.60\n33,111,MUG,1,5.25,5.25\n34,112,BTEA,4,3.20,12.80\n35,112,HTEA,1,4.10,4.10\n36,112,HON,2,7.50,15.00\n', 'invoices.csv': 'id,number,customer_id,invoice_date\n101,INV-001,1,2026-09-28\n102,INV-002,2,2026-09-29\n103,INV-003,3,2026-09-30\n104,INV-004,4,2026-10-01\n105,INV-005,5,2026-10-02\n106,INV-006,6,2026-10-03\n107,INV-007,7,2026-10-04\n108,INV-008,8,2026-10-05\n109,INV-009,1,2026-10-06\n110,INV-010,2,2026-10-07\n111,INV-011,5,2026-10-08\n112,INV-012,3,2026-10-09\n', 'products.csv': 'code,name,category,stock\nBTEA,Black tea,Drinks,30\nHTEA,Herbal tea,Drinks,18\nHON,Honey,Food,22\nSUG,Cane sugar,Food,40\nCHO,Dark chocolate,Food,35\nBIS,Biscuits,Food,28\nJAM,Strawberry jam,Food,16\nOAT,Oat drink,Drinks,32\nJUI,Apple juice,Drinks,26\nWAT,Mineral water,Drinks,60\nLEM,Lemonade,Drinks,30\nKET,Kettle,Kitchen,9\nPOT,Teapot,Kitchen,14\nPLT,Plate,Kitchen,36\nBWL,Bowl,Kitchen,24\nSPO,Spoon set,Kitchen,20\nTRY,Serving tray,Kitchen,11\nJAR,Storage jar,Kitchen,25\nNAP,Cotton napkins,Kitchen,18\nTWL,Tea towel,Kitchen,30\nFLT,Coffee filters,Kitchen,45\nTIN,Tea tin,Kitchen,15\nMAT,Table mats,Kitchen,16\nCAF,Coffee maker,Kitchen,7\nBOT,Water bottle,Kitchen,21\nSTR,Tea strainer,Kitchen,23\nSCL,Kitchen scale,Kitchen,8\n'}
for _name, _text in _csv_files.items():
    _ = Path(_name).write_text(_text, encoding="utf-8")


In [ ]:
#@title Recreate the previous lessons

######## Import our two tools ########

from asqueel import AsqueelDb, SqlDatabaseConfig


######## Choose SQLite in memory ########

model = SqlDatabaseConfig()
root = model.source.db(implementation="sqlite")
_ = root.connection(name=":memory:", implementation="sqlite")


######## Describe a product table ########

shop = root.packages().package("shop", sqlschema="shop")
product = shop.table("product", pkey="code")
_ = product.column("code", size=":10")
_ = product.column("name", size=":60")


######## Create that table ########

db = AsqueelDb(model)
print(db.migrate().commands)


######## Add our first product ########

products = db.table("shop.product")
products.insert({"code": "TEA", "name": "Tea"})
db.commit()


######## Read our product ########

products.query(columns="$code, $name").fetch()


######## Change its name ########

products.update({"name": "Green tea"}, pkey="TEA")
db.commit()


######## Check the update ########

products.query(columns="$code, $name").fetch()
db.commit()


######## Track category and stock ########

_ = product.column("category", size=":30")
_ = product.column("stock", dtype="I")


######## Load the updated model ########

db.reloadModel(model)


######## Add those database columns ########

print(db.migrate().commands)


######## Check our original row ########

products = db.table("shop.product")
products.query(columns="$name, $stock").fetch()
db.commit()


######## Give tea its stock ########

products.update({"category": "Drinks", "stock": 8}, pkey="TEA")
db.commit()


######## Read the new values ########

products.query(columns="$name, $category, $stock").fetch()
db.commit()


######## Add products from CSV ########

from io import StringIO
data = StringIO(
    "code,name,category,stock\n"
    "COF,Coffee,Drinks,24\n"
    "MUG,Mug,Kitchen,12\n"
)


######## Import those rows ########

products.sql_insert(data, format="csv")
db.commit()


######## List just the drinks ########

products.query(
    columns="$name, $stock",
    where="$category = :category", category="Drinks",
    order_by="$stock DESC",
).fetch()
db.commit()


## 1 — What we'll learn in this lesson

Welcome back! Here's what we'll learn in this lesson. Create customers, invoices and invoice rows, import our CSV files, and read fields through relationship paths.

```text
What we'll learn in this lesson
Create customers, invoices and invoice rows
Import products, dates, quantities and prices
Read fields through relationship paths

customer ← invoice ← invoice_row → product
```

The setup above has recreated the previous lessons in this notebook's kernel. Continue with the cells below.

Let's check our starting point. We already have three products.

In [ ]:
######## Check our starting point ########

products.query(columns="$code, $name").fetch()
db.commit()

## 2 — Add customers and their city

Let's describe our customers. Alongside the identifier and name, add their city.

In [ ]:
######## Describe our customers ########

customer = shop.table("customer", pkey="id")
_ = customer.column("id", dtype="I")
_ = customer.column("name", size=":60")
_ = customer.column("city", size=":60")

Let's create the customer table. Reload the model and migrate, keeping the existing products.

In [ ]:
######## Create the customer table ########

db.reloadModel(model)
print(db.migrate().commands)

## 3 — Give each invoice a date and customer

Let's describe a dated invoice. D means a date column. An invoice has its own date and number.

In [ ]:
######## Describe a dated invoice ########

invoice = shop.table("invoice", pkey="id")
_ = invoice.column("id", dtype="I")
_ = invoice.column("number", size=":20")
_ = invoice.column("invoice_date", dtype="D")

Let's connect invoices to customers. The stored customer identifier refers to the customer key. The inverse name invoices lets a customer reach its invoices.

In [ ]:
######## Connect invoices to customers ########

customer_id = invoice.column("customer_id", dtype="I")
_ = customer_id.relation(
    "shop.customer.id", foreign_key=True, relation_name="invoices",
)

## 4 — Give each row a quantity, price and total

Let's describe an invoice row. It needs quantity, unit price, and total. N declares numeric columns; twelve comma two describes precision and scale.

In [ ]:
######## Describe an invoice row ########

invoice_row = shop.table("invoice_row", pkey="id")
_ = invoice_row.column("id", dtype="I")
_ = invoice_row.column("quantity", dtype="I")
_ = invoice_row.column("unit_price", dtype="N", size="12,2")
_ = invoice_row.column("total", dtype="N", size="12,2")

Let's connect the row at both ends. The identifiers point to the invoice and product. Each inverse is called rows: the rows of one invoice, or the rows selling one product.

In [ ]:
######## Connect the row at both ends ########

invoice_id = invoice_row.column("invoice_id", dtype="I")
_ = invoice_id.relation(
    "shop.invoice.id", foreign_key=True, relation_name="rows",
)
product_code = invoice_row.column("product_code", size=":10")
_ = product_code.relation(
    "shop.product.code", foreign_key=True, relation_name="rows",
)

## 5 — Create the connected tables

Let's apply the new model. The migration creates invoices and rows, including their foreign keys.

In [ ]:
######## Apply the new model ########

db.reloadModel(model)
print(db.migrate().commands)

Let's get fresh table handles. We will populate these four tables from CSV files.

In [ ]:
######## Get fresh table handles ########

products = db.table("shop.product")
customers = db.table("shop.customer")
invoices = db.table("shop.invoice")
rows = db.table("shop.invoice_row")

## 6 — Download the sample CSV files

The four fictional CSV files are already in the temporary folder created above: `products.csv`, `customers.csv`, `invoices.csv` and `invoice_rows.csv`. Let's inspect and import them.

Let's inspect the row file. The CSV contains the keys, quantities, unit prices, and row totals. Each total is quantity times unit price.

In [ ]:
######## Inspect the row file ########

from pathlib import Path
print("\n".join(Path("invoice_rows.csv").read_text().splitlines()[:4]))

## 7 — Import the catalogue and customers

Let's expand the product catalogue. Pass the open file to SQL insert. The extension identifies CSV, and the model supplies the types. This bulk import skips Python triggers. Commit when finished.

In [ ]:
######## Expand the product catalogue ########

with open("products.csv", newline="", encoding="utf-8") as source:
    products.sql_insert(source)

db.commit()

Let's count the catalogue. Twenty-seven imported products plus our original three makes thirty.

In [ ]:
######## Count the catalogue ########

products.query().count()
db.commit()

Let's import customers and their cities. Use the same direct import for customers. Integer keys are converted automatically. Here are three customers, including Ada in Florence.

In [ ]:
######## Import customers and their cities ########

with open("customers.csv", newline="", encoding="utf-8") as source:
    customers.sql_insert(source)

db.commit()
customers.query(columns="$name, $city", limit=3).fetch()
db.commit()

## 8 — Import dated invoices

Let's import the invoices. Their customer keys must already exist. The model converts the ISO date strings automatically.

In [ ]:
######## Import the invoices ########

with open("invoices.csv", newline="", encoding="utf-8") as source:
    invoices.sql_insert(source)

db.commit()

## 9 — Import prices and totals

Let's import the invoice rows. SQL insert converts quantities to integers and amounts to Decimal. Totals are already in this CSV; importing does not calculate them.

In [ ]:
######## Import the invoice rows ########

with open("invoice_rows.csv", newline="", encoding="utf-8") as source:
    rows.sql_insert(source)

db.commit()

Let's check how much we imported. Twelve invoices and thirty-six invoice rows.

In [ ]:
######## Check how much we imported ########

invoices.query().count()
rows.query().count()
db.commit()

## 10 — Read the customer city and invoice date

Let's read invoices with their customers. Dollar invoice date is local. At customer I D follows the relation. Asqueel automatically names the returned columns after their paths.

In [ ]:
######## Read invoices with their customers ########

invoices.query(
    columns="$number, $invoice_date, "
            "@customer_id.name, @customer_id.city",
    limit=3,
).fetch()
db.commit()

## 11 — Read products, quantities and amounts

Let's inspect the first invoice. Product names come through the relation. Quantity, unit price, and total belong to the row: two teas at three fifty make seven.

In [ ]:
######## Inspect the first invoice ########

rows.query(
    columns="@product_code.name, "
            "$quantity, $unit_price, $total",
    where="$invoice_id = :invoice", invoice=101,
).fetch()
db.commit()

## 12 — Follow two relationships

Now follow a row to the customer. Go through the invoice, then the customer. The same path also reaches their city.

```text
invoice_row → invoice → customer
@invoice_id.@customer_id.city

invoice_row → invoice
@invoice_id.invoice_date

invoice_row → product
@product_code.name
```

Let's read across both connections. A single query returns invoice number, customer city, product name, and row total.

In [ ]:
######## Read across both connections ########

rows.query(
    columns="@invoice_id.number, "
            "@invoice_id.@customer_id.city, "
            "@product_code.name, $total",
    limit=3,
).fetch()
db.commit()

## 13 — Thank you — a complete small dataset

Thanks for following along! We imported a catalogue and dated invoices, imported row totals, and read fields through relationships. Next, we will use those same paths to filter and sort.

```text
Thank you!
30 products · 8 customers · 12 invoices · 36 rows
Row total = quantity × unit price
Next: filter and sort through relationships
```

## Check your result

Run this final cell to check the exercise.

In [ ]:
from decimal import Decimal
from datetime import date
assert products.query().count() == 30
assert customers.query().count() == 8
assert invoices.query().count() == 12
assert rows.query().count() == 36
assert all(isinstance(r["invoice_date"], date) for r in invoices.query(columns="$invoice_date").fetch())
assert all(r["total"] == r["quantity"] * r["unit_price"] for r in rows.query(columns="$quantity, $unit_price, $total").fetch())
db.commit()
print("Lesson checks passed.")
